# Phase 2 — LM Judge: Classify MedQA Clarifying Questions

Classifies each clarifying question from the Phase 1 results as **EPISTEMIC** or **ALEATORIC**
using the LLM judge with clinical few-shot examples.

Reads: `outputs/medqa/phase1_results.csv`  
Writes: `outputs/medqa/phase1_cq_classified.csv`

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../../").resolve()))

# ── Dataset config ────────────────────────────────────────────────────────
DATASET              = "medqa"
ROOT                 = Path("../../").resolve()
PROMPTS_DIR          = ROOT / "prompts"  / DATASET
OUTPUTS_DIR          = ROOT / "outputs"  / DATASET

JUDGE_INSTRUCTION    = PROMPTS_DIR / "judge_instruction.txt"
PHASE1_RESULTS_PATH  = OUTPUTS_DIR / "phase1_results.csv"
OUTPUT_PATH          = OUTPUTS_DIR / "phase1_cq_classified.csv"

# ── Run config ────────────────────────────────────────────────────────────
MODEL_ID             = "gemini-2.5-flash"
REQUEST_INTERVAL     = 1.0
REGENERATE           = False   # set True to re-classify from scratch
# ─────────────────────────────────────────────────────────────────────────

import logging
import pandas as pd
from IPython.display import display, Markdown

from src.utils import load_dotenv
from src.providers import GeminiProvider
from src.judge import LLMJudge, CSVBatchClassifier, FewShotExample

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s",
    datefmt="%H:%M:%S",
)

load_dotenv(ROOT / ".env")
print("Environment loaded.")

## Few-Shot Examples

One example per CLAMBER uncertainty subclass, adapted to the medical domain.  
These are hand-crafted clinical questions — **not** from the MedQA dataset (no leakage risk).

| Subclass | Label | Pattern |
|---|---|---|
| NK | EPISTEMIC | Model encounters an unfamiliar entity — asks to identify it |
| ICL | EPISTEMIC | Contradictory clinical picture — asks to resolve the contradiction |
| polysemy | ALEATORIC | Medically ambiguous term — multiple valid clinical meanings |
| co-reference | ALEATORIC | Ambiguous pronoun or referent in the patient's description |
| whom | ALEATORIC | Answer depends on patient-specific preference or priority |
| what | ALEATORIC | Underspecified request — multiple valid task interpretations |
| when | ALEATORIC | Ambiguous temporal scope in the description |
| where | ALEATORIC | Ambiguous spatial scope in the description |

In [ ]:
FEW_SHOT_EXAMPLES: list[FewShotExample] = [
    # ── EPISTEMIC: NK — model needs to identify an unfamiliar entity ──────
    FewShotExample(
        input="You mentioned having 'MCAS' — could you describe what symptoms it causes for you or what your doctor told you about it?",
        expected_output="EPISTEMIC",
        explanation=(
            "The model doesn't have enough context about this entity to reason "
            "clinically. There is a definite answer — once the patient explains "
            "it, the knowledge gap is fully and permanently resolved."
        ),
    ),
    # ── EPISTEMIC: ICL — contradictory clinical picture to resolve ────────
    FewShotExample(
        input="You described the rash as both 'spreading' and 'fading' — is it currently getting larger or is it clearing up?",
        expected_output="EPISTEMIC",
        explanation=(
            "The two descriptions contradict each other, making clinical "
            "categorisation impossible. There is one correct factual state right "
            "now — the model is resolving a factual contradiction, not a preference."
        ),
    ),
    # ── ALEATORIC: polysemy — term with multiple valid clinical meanings ───
    FewShotExample(
        input="When you say you feel 'weak', do you mean you have no energy and feel fatigued, or that you have actual muscle weakness and difficulty lifting things?",
        expected_output="ALEATORIC",
        explanation=(
            "'Weak' has two clinically distinct meanings (fatigue vs true motor "
            "weakness) that point to completely different differentials. No external "
            "fact can resolve which meaning the patient intends — only the patient can."
        ),
    ),
    # ── ALEATORIC: co-reference — ambiguous referent ──────────────────────
    FewShotExample(
        input="When you said 'it started after the procedure', are you referring to the chest pain or the shortness of breath?",
        expected_output="ALEATORIC",
        explanation=(
            "The pronoun 'it' could validly refer to either symptom. No external "
            "fact can determine which one the patient meant — only the patient's "
            "own context resolves this."
        ),
    ),
    # ── ALEATORIC: whom — depends on patient-specific priority ────────────
    FewShotExample(
        input="Which aspect of your recovery matters most to you — getting back to work quickly, minimising pain, or avoiding surgery?",
        expected_output="ALEATORIC",
        explanation=(
            "The answer depends entirely on this individual patient's values and "
            "priorities. No clinical fact or external knowledge can determine "
            "their personal preference — it is irreducibly patient-specific."
        ),
    ),
    # ── ALEATORIC: what — underspecified request ──────────────────────────
    FewShotExample(
        input="When you ask about treatment options, are you looking for information about medications, surgical approaches, or lifestyle changes?",
        expected_output="ALEATORIC",
        explanation=(
            "The request is underspecified — multiple valid interpretations exist "
            "and the correct path depends entirely on what the patient wants, "
            "not on any clinical fact."
        ),
    ),
    # ── ALEATORIC: when — ambiguous temporal scope ────────────────────────
    FewShotExample(
        input="When you say your symptoms are 'intermittent', do you mean they come and go throughout the day, or that you have symptom-free periods lasting weeks?",
        expected_output="ALEATORIC",
        explanation=(
            "Two valid temporal interpretations exist, each with different clinical "
            "significance. Only the patient knows which pattern applies — "
            "no external fact resolves this."
        ),
    ),
    # ── ALEATORIC: where — ambiguous spatial scope ────────────────────────
    FewShotExample(
        input="When you say the pain is 'everywhere', do you mean it is diffuse throughout your abdomen, or that it shifts between different locations?",
        expected_output="ALEATORIC",
        explanation=(
            "Two spatially distinct clinical patterns (diffuse vs migratory pain) "
            "are both plausible readings. Only the patient can clarify which "
            "pattern they actually experience."
        ),
    ),
]

print(f"Few-shot examples: {len(FEW_SHOT_EXAMPLES)}")
for ex in FEW_SHOT_EXAMPLES:
    print(f"  [{ex.expected_output}] {ex.input[:80]}")

## Smoke Test

In [ ]:
provider = GeminiProvider(model_id=MODEL_ID)

judge = LLMJudge(
    provider=provider,
    instructions_path=JUDGE_INSTRUCTION,
    few_shot_examples=FEW_SHOT_EXAMPLES,
    label_parser=lambda text: text.strip().upper(),
)

# Smoke test with a question not in few-shot examples
smoke = judge.evaluate("Have you been diagnosed with hypertension or any other heart condition before?")
print(f"Smoke test → label={smoke.label}, error={smoke.error}")
assert smoke.label == "EPISTEMIC", f"Unexpected smoke test label: {smoke.label}"
print("Smoke test passed.")

## Load Phase 1 Results

In [ ]:
phase1_df = pd.read_csv(PHASE1_RESULTS_PATH)

# Keep only valid, unblocked rows with a real clarifying question
work_df = phase1_df[
    (~phase1_df["was_blocked"])
    & (phase1_df["clarifying_question"].notna())
    & (phase1_df["clarifying_question"].str.strip() != "")
    & (phase1_df["clarifying_question"] != "BLOCKED")
].copy()

print(f"Phase 1 rows total:  {len(phase1_df)}")
print(f"Usable CQs:          {len(work_df)}")
print()
display(work_df[["id", "ehr_summary", "clarifying_question"]].head(5))

## Classify Clarifying Questions

In [ ]:
# Save work_df as input CSV for the batch classifier
INPUT_CSV = OUTPUTS_DIR / "phase1_cq_input.csv"
work_df[["id", "clarifying_question"]].to_csv(INPUT_CSV, index=False)
print(f"Input CSV saved: {INPUT_CSV}")

if REGENERATE and OUTPUT_PATH.exists():
    OUTPUT_PATH.unlink()
    print("Deleted existing output — regenerating.")

classifier = CSVBatchClassifier(
    judge=judge,
    input_csv=INPUT_CSV,
    output_csv=OUTPUT_PATH,
    question_column="clarifying_question",
    id_column="id",
    delay_between_calls=REQUEST_INTERVAL,
)
classifier.run()
print(f"Classification complete → {OUTPUT_PATH}")

## Results Summary

In [ ]:
classified_df = pd.read_csv(OUTPUT_PATH)

# Merge with phase1 results for full context
merged = phase1_df.merge(
    classified_df[["id", "label", "latency_seconds", "error"]].rename(
        columns={"label": "cq_type", "latency_seconds": "judge_latency"}
    ),
    on="id", how="left",
)

valid_labels = {"EPISTEMIC", "ALEATORIC"}
classified = merged[merged["cq_type"].isin(valid_labels)]

print(f"Total classified: {len(classified_df)}")
print(f"Valid labels:     {len(classified)}")
print(f"Invalid/errors:   {len(classified_df) - len(classified)}")
print()
print("Label distribution:")
print(classified["cq_type"].value_counts().to_string())
print()

# Correctness by CQ type
print("Updated correctness by CQ type:")
display(
    classified.groupby("cq_type")[["is_correct_preliminary", "is_correct_updated", "confidence_delta"]]
    .agg({"is_correct_preliminary": "mean", "is_correct_updated": "mean", "confidence_delta": "mean"})
    .round(3)
)

print()
print("Sample classifications:")
display(classified[["id", "cq_type", "clarifying_question", "is_correct_updated", "confidence_delta"]].head(15))